# Experiment 7 analysis

Reads the consolidated CSVs under `data/outputs/Experiment 7/_analysis/`. Run-level tables come from `runs_cache.csv`; score distributions and GMM analysis come from `evaluation_scores.csv`.

The notebook does not read raw experiment output folders.


In [ ]:
from __future__ import annotations

import math
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "outputs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
NOTEBOOK_DIR = PROJECT_ROOT / "notebooks and aux scripts"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from hardware_equivalence import normalize_latency

pd.set_option("display.float_format", lambda v: f"{v:.4f}")
plt.rcParams["figure.dpi"] = 110
from scipy import stats
from sklearn.mixture import GaussianMixture

ALL_MODELS = [
    "patchcore", "padim", "subspacead", "stfpm",
    "csflow", "draem", "rd4ad",
]

CROISSANT_CATEGORIES = ["croissant_dataset"]

EXP_ROOT = PROJECT_ROOT / "data" / "outputs" / "Experiment 7"
ANALYSIS_DIR = EXP_ROOT / "_analysis"
RUNS_CSV = ANALYSIS_DIR / "runs_cache.csv"
SCORES_CSV = ANALYSIS_DIR / "evaluation_scores.csv"

print(f"Runs CSV: {RUNS_CSV}")
print(f"Scores CSV: {SCORES_CSV}")


In [ ]:
df = pd.read_csv(RUNS_CSV)
score_df = pd.read_csv(SCORES_CSV, low_memory=False)
normalize_latency(df)
MODELS_PRESENT = sorted(df["model"].dropna().unique()) if not df.empty else []
print(f"Loaded {len(df)} consolidated run rows across models: {MODELS_PRESENT}")
print(f"Loaded {len(score_df)} consolidated evaluation score rows.")


## 1. Runs per model

Count of output folders found in `Experiment 7/` for each model. Only one category (`croissant_dataset`) was run.

In [ ]:
runs_per_cat = (
    df.groupby(["category", "model"]).size().unstack(fill_value=0)
    .reindex(index=CROISSANT_CATEGORIES, columns=ALL_MODELS, fill_value=0)
)
runs_per_cat.index.name = "category"
runs_per_cat.loc["TOTAL"] = runs_per_cat.sum(axis=0)
runs_per_cat

## 2. Per-model summary table

Headline numbers for each run. Because the croissant capture is unlabeled, the only "detection" figure is the **NG rate** — the fraction of evaluation frames that the model flagged as anomalous at its calibrated threshold.

$$
\text{NG rate} = \frac{\#\{\,\text{frames with}\ \widehat{y}_t = 1\,\}}{\#\{\,\text{evaluation frames}\,\}}
$$

Threshold (and so NG rate) is set by the POT/quantile rule used during streaming; warm-up and calibration frames are excluded.

In [ ]:
SUMMARY_COLS = [
    "threshold_mode", "threshold_value",
    "warmup_frames", "calibration_frames", "streaming_frames",
    "n_predicted_anomalies", "ng_rate",
    "mean_score", "median_score", "p95_score", "p99_score", "max_score",
    "mean_latency_ms", "throughput_fps",
]

summary = (
    df.sort_values("model")
    .set_index("model")[SUMMARY_COLS]
)
summary

## 3. Score distribution per model

Histogram of per-frame anomaly scores over the evaluation phase (warm-up and threshold-calibration excluded). The vertical dashed line marks the calibrated threshold used at inference; everything to the right of it counted toward the NG rate.

**What to look for** — a clean detector on an unlabeled stream typically produces a **bimodal** score histogram: a dense "OK" mode on the left, a sparse "NG" mode on the right, with an empty valley in between (the threshold should fall inside that valley). A unimodal distribution suggests the model cannot separate normal vs anomalous appearance on this dataset.

In [ ]:
def load_evaluation_scores(experiment: str, model: str | None = None) -> np.ndarray:
    sub = score_df[score_df["experiment"] == experiment]
    if model is not None:
        sub = sub[sub["model"] == model]
    return sub.sort_values("idx")["score"].to_numpy(dtype=float)


HIST_COLS = 2
n_models = len(MODELS_PRESENT)
nrows = math.ceil(n_models / HIST_COLS) if n_models else 1
fig, axes = plt.subplots(
    nrows, HIST_COLS,
    figsize=(HIST_COLS * 5.5, nrows * 3.2),
    squeeze=False,
)
fig.suptitle("Score distribution per model ? evaluation phase", fontsize=12)

for i, model in enumerate(MODELS_PRESENT):
    ax = axes[i // HIST_COLS][i % HIST_COLS]
    row = df[df["model"] == model].iloc[0]
    scores = load_evaluation_scores(row["experiment"], model)
    if scores.size == 0:
        ax.set_visible(False)
        continue
    upper = float(np.percentile(scores, 99.5))
    lower = float(np.min(scores))
    ax.hist(scores, bins=80, range=(lower, max(upper, lower + 1e-9)),
            color="tab:blue", alpha=0.75, edgecolor="white", linewidth=0.3)
    thr = row["threshold_value"]
    if pd.notna(thr):
        ax.axvline(float(thr), color="black", lw=0.9, ls="--",
                   label=f"threshold = {float(thr):.3g}")
    ng_rate = row["ng_rate"]
    ax.set_title(f"{model}  ?  NG rate = {ng_rate*100:.2f}%  (n={scores.size})", fontsize=10)
    ax.set_xlabel("anomaly score", fontsize=9)
    ax.set_ylabel("frame count", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(alpha=0.25)

for j in range(n_models, nrows * HIST_COLS):
    axes[j // HIST_COLS][j % HIST_COLS].set_visible(False)

fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


### 3b. Same histograms on log-y

Log-scaled y-axis to make the *tail* (candidate NG cluster) visible — raw counts of the OK mode swamp the histogram on a linear axis.

In [ ]:
fig, axes = plt.subplots(
    nrows, HIST_COLS,
    figsize=(HIST_COLS * 5.5, nrows * 3.2),
    squeeze=False,
)
fig.suptitle("Score distribution per model (log y) \u2014 evaluation phase", fontsize=12)

for i, model in enumerate(MODELS_PRESENT):
    ax = axes[i // HIST_COLS][i % HIST_COLS]
    row = df[df["model"] == model].iloc[0]
    scores = load_evaluation_scores(row["experiment"], model)
    if scores.size == 0:
        ax.set_visible(False)
        continue
    upper = float(np.percentile(scores, 99.9))
    lower = float(np.min(scores))
    ax.hist(scores, bins=80, range=(lower, max(upper, lower + 1e-9)),
            color="tab:orange", alpha=0.75, edgecolor="white", linewidth=0.3)
    ax.set_yscale("log")
    thr = row["threshold_value"]
    if pd.notna(thr):
        ax.axvline(float(thr), color="black", lw=0.9, ls="--",
                   label=f"threshold = {float(thr):.3g}")
    ax.set_title(f"{model}", fontsize=10)
    ax.set_xlabel("anomaly score", fontsize=9)
    ax.set_ylabel("log count", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=8, loc="upper right")
    ax.grid(alpha=0.25, which="both")

for j in range(n_models, nrows * HIST_COLS):
    axes[j // HIST_COLS][j % HIST_COLS].set_visible(False)

fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


## 4. Streaming scores over frame order

Per-frame anomaly score across the evaluation phase. Without labels we colour every point grey and only mark frames that crossed the threshold ("predicted NG") in red. Useful to see whether NG predictions cluster in time (drift / camera issue) or appear sporadically (genuine defects).

In [ ]:
def load_streaming_records(experiment: str, model: str | None = None) -> pd.DataFrame:
    sub = score_df[score_df["experiment"] == experiment]
    if model is not None:
        sub = sub[sub["model"] == model]
    return sub.sort_values("idx")


SCATTER_COLS = 2
fig, axes = plt.subplots(
    nrows, SCATTER_COLS,
    figsize=(SCATTER_COLS * 5.5, nrows * 3.0),
    squeeze=False,
)
fig.suptitle("Streaming scores vs frame order ? predicted NG highlighted", fontsize=12)

for i, model in enumerate(MODELS_PRESENT):
    ax = axes[i // SCATTER_COLS][i % SCATTER_COLS]
    row = df[df["model"] == model].iloc[0]
    recs = load_streaming_records(row["experiment"], model)
    if recs.empty:
        ax.set_visible(False)
        continue
    idx = recs["idx"].to_numpy(dtype=int)
    scores = recs["score"].to_numpy(dtype=float)
    preds = recs["pred_label"].fillna(-1).to_numpy(dtype=int)
    ng = preds == 1
    ax.scatter(idx[~ng], scores[~ng], s=4, c="#888888", alpha=0.40, label="predicted OK")
    if ng.any():
        ax.scatter(idx[ng], scores[ng], s=10, c="tab:red", alpha=0.85, label="predicted NG")
    thr = row["threshold_value"]
    if pd.notna(thr):
        ax.axhline(float(thr), color="black", lw=0.7, ls="--", label=f"thr={float(thr):.3g}")
    upper = float(np.percentile(scores, 99.5))
    ax.set_ylim(top=max(upper, float(thr) * 1.1 if pd.notna(thr) else upper))
    ax.set_title(f"{model}", fontsize=10)
    ax.set_xlabel("frame order (evaluation idx)", fontsize=9)
    ax.set_ylabel("score", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=7, loc="upper right")
    ax.grid(alpha=0.25)

for j in range(n_models, nrows * SCATTER_COLS):
    axes[j // SCATTER_COLS][j % SCATTER_COLS].set_visible(False)

fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


## 5. Unsupervised separation proxies

Without labels we cannot compute AUROC or F1, but we can ask: *does the model produce two well-separated score modes?* That is the working assumption behind **Yu et al. (2023), "Anomaly Detection with Score Distribution Discrimination"** — they train detectors with an **Overlap Loss** that pushes the OK and NG score histograms apart, and use the same overlap quantity as an unsupervised quality signal at inference. We compute two label-free proxies inspired by that view:

### 5a. Bimodality coefficient (SAS / Pfister et al. 2013)

Closed-form statistic from sample skew $g$ and excess kurtosis $k$:

$$
\mathrm{BC} = \frac{g^{2} + 1}{k + \frac{3(n-1)^{2}}{(n-2)(n-3)}}
$$

A perfect uniform distribution gives $\mathrm{BC} = 5/9 \approx 0.555$; values **above** that threshold are commonly read as evidence of bimodality (multi-cluster structure in the score histogram). Cheap, requires no model fitting.

### 5b. Two-component GMM standardized separation

Fit a 2-component Gaussian mixture to the (log-)scores. Let $(\mu_1, \sigma_1)$ and $(\mu_2, \sigma_2)$ be the two component parameters with $\mu_2 > \mu_1$ (low mode = OK candidate, high mode = NG candidate). Report:

$$
D = \frac{\mu_2 - \mu_1}{\sqrt{\tfrac{1}{2}(\sigma_1^{2} + \sigma_2^{2})}}
\quad \text{and} \quad
\pi_{\text{NG}} = \text{mixture weight of the high mode.}
$$

$D$ is a Cohen-style *d* between the two latent clusters: $D \gtrsim 2$ means the modes are well separated (low overlap, useful detector); $D \lesssim 1$ means the score histogram looks like one mode and the threshold is essentially cutting noise. $\pi_{\text{NG}}$ is a *model-internal* estimate of the NG rate that does **not** depend on the operational threshold — comparing $\pi_{\text{NG}}$ against the threshold-driven NG rate from §2 tells us whether the threshold is too aggressive/conservative for what the score distribution actually contains.

In [ ]:
def bimodality_coefficient(x: np.ndarray) -> float:
    n = x.size
    if n < 4:
        return float("nan")
    g = float(stats.skew(x, bias=False))
    k = float(stats.kurtosis(x, fisher=True, bias=False))  # excess kurtosis
    correction = 3.0 * (n - 1) ** 2 / ((n - 2) * (n - 3))
    denom = k + correction
    if denom <= 0:
        return float("nan")
    return (g * g + 1.0) / denom


def gmm_two_component(x: np.ndarray, *, log_transform: bool = True, seed: int = 0) -> dict:
    """Fit GMM(k=2) and return standardized separation D and mixture weights."""
    if x.size < 50 or np.std(x) == 0:
        return {"D": float("nan"), "pi_low": float("nan"), "pi_high": float("nan"),
                "mu_low": float("nan"), "mu_high": float("nan"),
                "sigma_low": float("nan"), "sigma_high": float("nan")}
    data = x.astype(float)
    if log_transform:
        shift = max(0.0, -float(np.min(data)) + 1e-6)
        data = np.log1p(data + shift)
    gmm = GaussianMixture(n_components=2, covariance_type="full", random_state=seed,
                         n_init=4, max_iter=300).fit(data.reshape(-1, 1))
    means = gmm.means_.ravel()
    sigmas = np.sqrt(gmm.covariances_.ravel())
    weights = gmm.weights_.ravel()
    order = np.argsort(means)  # low, high
    mu_lo, mu_hi = means[order]
    s_lo, s_hi = sigmas[order]
    w_lo, w_hi = weights[order]
    D = (mu_hi - mu_lo) / math.sqrt(0.5 * (s_lo ** 2 + s_hi ** 2)) if (s_lo + s_hi) > 0 else float("nan")
    return {"D": float(D), "pi_low": float(w_lo), "pi_high": float(w_hi),
            "mu_low": float(mu_lo), "mu_high": float(mu_hi),
            "sigma_low": float(s_lo), "sigma_high": float(s_hi)}


rows_sep = []
for model in MODELS_PRESENT:
    row = df[df["model"] == model].iloc[0]
    scores = load_evaluation_scores(row["experiment"], model)
    bc = bimodality_coefficient(scores)
    gmm_res = gmm_two_component(scores)
    rows_sep.append({
        "model": model,
        "n_scores": int(scores.size),
        "bimodality_coef": bc,
        "bimodal?": "\u2713" if (not math.isnan(bc) and bc > 5 / 9) else "\u00d7",
        "gmm_D": gmm_res["D"],
        "gmm_pi_high": gmm_res["pi_high"],
        "ng_rate (§2)": float(row["ng_rate"]),
    })

sep_df = pd.DataFrame(rows_sep).set_index("model")
sep_df


### 5c. GMM(k=2) overlay on the score histogram

Visual check: the two fitted Gaussian components are plotted on top of the score histogram (KDE-style). When $D \gtrsim 2$ the two bells should sit on the OK mode and the NG tail respectively; when $D$ is small the two components collapse onto the same mode and the fit is meaningless.

In [ ]:
fig, axes = plt.subplots(
    nrows, HIST_COLS,
    figsize=(HIST_COLS * 5.5, nrows * 3.2),
    squeeze=False,
)
fig.suptitle("GMM(k=2) fit over log-scores (Yu et al. 2023 separation view)", fontsize=12)

for i, model in enumerate(MODELS_PRESENT):
    ax = axes[i // HIST_COLS][i % HIST_COLS]
    row = df[df["model"] == model].iloc[0]
    scores = load_evaluation_scores(row["experiment"], model)
    if scores.size == 0:
        ax.set_visible(False)
        continue
    shift = max(0.0, -float(np.min(scores)) + 1e-6)
    log_s = np.log1p(scores + shift)
    gmm = GaussianMixture(n_components=2, covariance_type="full", random_state=0,
                         n_init=4, max_iter=300).fit(log_s.reshape(-1, 1))
    means = gmm.means_.ravel(); sigmas = np.sqrt(gmm.covariances_.ravel()); weights = gmm.weights_.ravel()
    order = np.argsort(means)
    lo_x = float(np.min(log_s)); hi_x = float(np.percentile(log_s, 99.5))
    grid = np.linspace(lo_x, hi_x, 400)
    ax.hist(log_s, bins=80, range=(lo_x, hi_x), density=True,
            color="tab:blue", alpha=0.45, edgecolor="white", linewidth=0.3)
    colors = ["tab:green", "tab:red"]
    for j, idx in enumerate(order):
        pdf = weights[idx] * stats.norm.pdf(grid, means[idx], sigmas[idx])
        ax.plot(grid, pdf, color=colors[j], lw=1.5,
                label=f"comp {j+1}: μ={means[idx]:.2f}, σ={sigmas[idx]:.2f}, π={weights[idx]:.2f}")
    D = (means[order[1]] - means[order[0]]) / math.sqrt(0.5 * (sigmas[order[0]] ** 2 + sigmas[order[1]] ** 2))
    ax.set_title(f"{model}  \u2014  D = {D:.2f}", fontsize=10)
    ax.set_xlabel("log(1 + score + shift)", fontsize=9)
    ax.set_ylabel("density", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.legend(fontsize=7, loc="upper right")
    ax.grid(alpha=0.25)

for j in range(n_models, nrows * HIST_COLS):
    axes[j // HIST_COLS][j % HIST_COLS].set_visible(False)

fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


## References

- **Yu, M., Wang, T., Zhang, R., Ding, K., Zhao, Y., He, J., Hu, T., & Cheng, J. (2023).** *Anomaly Detection with Score Distribution Discrimination*. KDD '23. [arXiv:2306.14403](https://arxiv.org/abs/2306.14403). — introduces *Overlap loss* and motivates evaluating an anomaly detector by how separated its normal vs anomaly score distributions are; their Section 2 explicitly discusses estimating these distributions when anomaly labels are scarce or absent.
- **Pfister, R., Schwarz, K. A., Janczyk, M., Dale, R., & Freeman, J. (2013).** *Good things peak in pairs: a note on the bimodality coefficient*. Frontiers in Psychology, 4, 700. — closed-form bimodality coefficient with the $5/9$ uniform-distribution cutoff used in §5a.
- **Hartigan, J. A., & Hartigan, P. M. (1985).** *The dip test of unimodality*. The Annals of Statistics, 13(1), 70–84. — non-parametric alternative for testing whether a one-dimensional sample is unimodal; useful as a complement to BC when sample size is large.